# Apache Parquet - Rust

All 11 Rust examples from [docs/parquet.md](https://platob.github.io/yggdryl/parquet/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

## Arrow batch reads and writes

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, Url};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");
let schema = field.into_arrow_schema()?;
let batch = |ids: Vec<i64>, symbols: Vec<Option<&str>>| {
    RecordBatch::try_new(
        Arc::clone(&schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(symbols)),
        ],
    )
};

// The name decides Parquet; the methods name the write intent.
let mut handle =
    Buffer::new().with_media_type(Url::from_str("file:///trades.parquet")?.media_type());
let options = handle.record_options()?;
handle.overwrite_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![1, 2], vec![Some("AAPL"), Some("MSFT")])?],
    ),
    &options,
)?;
handle.append_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![3], vec![Some("GOOG")])?],
    ),
    &options,
)?;
handle.merge_arrow_reader(
    yggdryl::arrow::batch_reader(
        Arc::clone(&schema),
        [batch(vec![2, 4], vec![Some("NVDA"), None])?],
    ),
    &options.clone().with_merge_by_names(["id"]),
)?;

let rows = handle
    .read_arrow_reader(&options)?
    .map(|batch| batch.map(|batch| batch.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(rows, 4);

### Dimensions and opened sessions

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::generic::Holder;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let schema = field.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;
let mut handle = Holder::buffer(Buffer::new().with_media_type(MimeType::PARQUET.into()));
let options = handle.record_options()?;
handle.overwrite_arrow_reader(yggdryl::arrow::batch_reader(schema, [batch]), &options)?;

handle.open()?;
assert_eq!(handle.read_arrow_field(&options)?, field);
assert_eq!((handle.row_size()?, handle.column_size()?), (2, 1));
handle.close()?;

## Column pushdown

In [ ]:
use std::sync::Arc;

use arrow_array::{Float64Array, Int64Array, RecordBatch, RecordBatchReader, StringArray};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let stored = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.required_field("symbol"),
    DataType::Float64.required_field("price"),
    DataType::Utf8.required_field("venue"),
])?
.required_field("row");
let arrow_schema = stored.into_arrow_schema()?;

let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2])),
        Arc::new(StringArray::from(vec!["AAPL", "MSFT"])),
        Arc::new(Float64Array::from(vec![1.5, 2.5])),
        Arc::new(StringArray::from(vec!["XNAS", "XNAS"])),
    ],
)?;

let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
let options = media.record_options()?;
media.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;

// Two of the four columns, named by a root Field of its own.
let wanted = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Float64.required_field("price"),
])?
.required_field("row");

let projected = media.read_arrow_reader(&options.clone().with_field(wanted))?;
assert_eq!(projected.schema().fields().len(), 2);
let read = projected.collect::<Result<Vec<_>, _>>()?;
assert_eq!(read[0].num_columns(), 2);

// The file is unchanged: it still stores all four.
assert_eq!(media.read_arrow_schema()?.fields().len(), 4);

## Options

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, Level, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = field.clone().into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from((0..1_000).collect::<Vec<i64>>()))],
)?;

// Parquet's own settings and the shared ones are flat fields on one struct.
let options = ParquetOptions::new()
    .with_max_row_group_size(4_096)
    .with_key_value("iceberg.schema-id", "7")
    .with_batch_size(256)
    .with_root_name("trade");

assert_eq!(options.max_row_group_size, 4_096);
assert_eq!(
    options.key_value_metadata,
    [("iceberg.schema-id".to_owned(), "7".to_owned())]
);
assert_eq!(options.batch_size(), Some(256));
assert_eq!(options.root_name(), "trade");
assert!(!options.safe());

// Unused here: Parquet compresses pages itself.
assert_eq!(options.level, Level::DEFAULT);

let mut media =
    Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into())).with_options(options);
let call_options = media.record_options()?;
media.overwrite_arrow_reader(
    arrow::batch_reader(arrow_schema, [batch]),
    &call_options,
)?;

// batch_size bounds the reader, so no batch holds all 1,000 rows.
let rows: Vec<usize> = media
    .read_arrow_reader(&call_options)?
    .collect::<Result<Vec<_>, _>>()?
    .iter()
    .map(arrow_array::RecordBatch::num_rows)
    .collect();
assert_eq!(rows.iter().sum::<usize>(), 1_000);
assert!(rows.iter().all(|count| *count <= 256), "{rows:?}");

// The root name names the Field recovered from the footer.
assert_eq!(media.read_arrow_field(&call_options)?.name(), "trade");

// A declared schema is returned as-is, so an empty handle answers without a footer.
let declared =
    Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into())).with_field(field.clone());
let declared_options = declared.record_options()?;
assert_eq!(declared.read_arrow_field(&declared_options)?, field);

## Compression

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use parquet::basic::Compression;
use yggdryl::arrow;
use yggdryl::generic::IORecordOptions;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let ids: Vec<i64> = (0..4_000).collect();
let symbols: Vec<Option<&str>> = ids.iter().map(|_| Some("AAPL")).collect();
let arrow_schema = field.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(ids)),
        Arc::new(StringArray::from(symbols)),
    ],
)?;

let mut sizes = Vec::new();
for compression in [
    Compression::UNCOMPRESSED,
    Compression::SNAPPY,
    Compression::ZSTD(Default::default()),
] {
    // One batch per read, so the comparison is not split by the default bound.
    let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()))
        .with_options(
            ParquetOptions::new()
                .with_compression(compression)
                .with_batch_size(batch.num_rows()),
        );
    let options = media.record_options()?;
    media.overwrite_arrow_reader(
        arrow::batch_reader(Arc::clone(&arrow_schema), [batch.clone()]),
        &options,
    )?;

    // Nothing on the read side names the compression: the footer records it.
    let read = media
        .read_arrow_reader(&options)?
        .collect::<Result<Vec<_>, _>>()?;
    assert_eq!(read, [batch.clone()], "{compression:?}");
    sizes.push(media.handle().size());
}

assert!(sizes[0] > sizes[1] && sizes[0] > sizes[2], "{sizes:?}");

## Coded handles are rejected

In [ ]:
use arrow_array::RecordBatch;
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, Url};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

// The name declares gzip over the Parquet file.
let url = Url::from_str("file:///trades.parquet.gz")?;
let mut media = Parquet::new(Buffer::new().with_media_type(url.media_type()));

let empty = arrow::batch_reader(field.into_arrow_schema()?, std::iter::empty::<RecordBatch>());
let options = media.record_options()?;
let message = media
    .overwrite_arrow_reader(empty, &options)
    .unwrap_err()
    .to_string();
assert!(message.contains("parquet compresses"), "{message}");
assert!(message.contains("ParquetOptions::compression"), "{message}");

// Nothing was published.
assert!(media.handle().is_empty());

## Field identifiers

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([
    DataType::Int64.required_field("id").with_parquet_field_id(1),
    DataType::Utf8.nullable_field("symbol").with_parquet_field_id(2),
])?
.required_field("row");

let arrow_schema = field.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![
        Arc::new(Int64Array::from(vec![1])),
        Arc::new(StringArray::from(vec![Some("AAPL")])),
    ],
)?;

let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
let options = media.record_options()?;
media.overwrite_arrow_reader(arrow::batch_reader(arrow_schema, [batch]), &options)?;

// The ids went into the file, so the Arrow schema carries them back.
let schema = media.read_arrow_schema()?;
assert_eq!(
    schema.field(0).metadata().get("PARQUET:field_id"),
    Some(&"1".to_owned())
);

// And the recovered Field answers by id rather than by position.
let recovered = media.read_arrow_field(&options)?;
assert_eq!(recovered.fields()[0].parquet_field_id()?, Some(1));
assert_eq!(recovered.fields()[1].parquet_field_id()?, Some(2));

## Footer statistics

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::{Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType, Scalar};

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?.required_field("row");
let schema = field.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2, 3, 4])),
        Arc::new(StringArray::from(vec![Some("AAPL"), None, Some("MSFT"), None])),
    ],
)?;
let mut media = Parquet::new(
    Buffer::new().with_media_type(MimeType::PARQUET.into()),
).with_options(
    ParquetOptions::new()
        .with_max_row_group_size(2)
        .with_key_value("writer", "rust"),
);
let options = media.record_options()?;
media.overwrite_arrow_reader(arrow::batch_reader(schema, [batch]), &options)?;

let statistics = IOMedia::read_parquet_statistics(&media)?;
assert_eq!(statistics.num_rows, 4);
assert_eq!(statistics.row_groups.len(), 2);
assert_eq!(statistics.null_count("symbol"), Some(2));
let native = Scalar::from(statistics);
assert_eq!(native.get_key_str("num_rows").and_then(Scalar::as_i64), Some(4));

## Geospatial and variant columns

In [ ]:
use std::sync::Arc;

use arrow_array::{BinaryArray, Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

fn wkb_point(x: f64, y: f64) -> Vec<u8> {
    let mut bytes = vec![1u8];
    bytes.extend_from_slice(&1u32.to_le_bytes());
    bytes.extend_from_slice(&x.to_le_bytes());
    bytes.extend_from_slice(&y.to_le_bytes());
    bytes
}

let field = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::geometry(None)?.nullable_field("shape"),
])?
.required_field("row");
let schema = field.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&schema),
    vec![
        Arc::new(Int64Array::from(vec![1, 2, 3])),
        Arc::new(BinaryArray::from_opt_vec(vec![
            Some(&wkb_point(1.0, 2.0)[..]),
            None,
            Some(&wkb_point(-3.0, 7.0)[..]),
        ])),
    ],
)?;
let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
let options = media.record_options()?;
media.overwrite_arrow_reader(arrow::batch_reader(schema, [batch]), &options)?;

let statistics = media.read_statistics()?;
let columns = &statistics.row_groups[0].columns;
let id = columns.iter().find(|column| column.path == "id").unwrap();
let shape = columns.iter().find(|column| column.path == "shape").unwrap();
assert!(id.min_bytes.is_some() && id.max_bytes.is_some());
assert!(shape.min_bytes.is_none() && shape.max_bytes.is_none());
assert_eq!(shape.null_count, Some(1));

let geospatial = shape.geospatial.as_ref().unwrap();
let bounds = geospatial.bounding_box.unwrap();
assert_eq!(
    (bounds.xmin, bounds.xmax, bounds.ymin, bounds.ymax),
    (-3.0, 1.0, 2.0, 7.0)
);
assert_eq!(geospatial.geometry_types, vec![1]);
assert_eq!(
    IOMedia::read_parquet_geospatial_statistics(&media, "shape")?,
    *geospatial,
);

## The handle underneath

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOBase, IOMedia};
use yggdryl::parquet::{self, Parquet, ParquetOptions};
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");
let arrow_schema = field.into_arrow_schema()?;
let batch = RecordBatch::try_new(
    Arc::clone(&arrow_schema),
    vec![Arc::new(Int64Array::from(vec![1, 2]))],
)?;

// The free functions take a handle and options; nothing is bound.
let options = ParquetOptions::new();
let mut handle = Buffer::new().with_media_type(MimeType::PARQUET.into());
parquet::overwrite_batch_reader(
    &mut handle,
    arrow::batch_reader(arrow_schema, [batch]),
    &options,
)?;

assert_eq!(parquet::read_arrow_schema(&handle)?.fields().len(), 1);
assert_eq!(parquet::read_field(&handle, &options)?.name(), "row");
assert_eq!(parquet::read_batch_reader(&handle, None, &options)?.count(), 1);
assert_eq!(parquet::read_statistics(&handle)?.num_rows, 2);

// A Parquet is also the bytes it encodes, magic bytes included.
let mut media = Parquet::new(handle);
assert_eq!(media.read_range(0, 4)?, *b"PAR1");

// open caches the footer, close releases it.
assert!(!media.opened());
media.open()?;
assert!(media.opened());
assert_eq!(media.read_statistics()?.num_rows, 2);
assert_eq!(media.row_size()?, 2);
assert_eq!(media.column_size()?, 1);
media.close()?;
assert!(!media.opened());

In [ ]:
use arrow_array::{RecordBatch, RecordBatchReader};
use yggdryl::arrow;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::parquet::Parquet;
use yggdryl::{DataType, MimeType};

let field = DataType::from_fields([DataType::Int64.required_field("id")])?.required_field("row");

// Nothing has been written, so there is nothing to read.
let empty = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()))
    .with_field(field.clone());
let options = empty.record_options()?;
let reader = empty.read_arrow_reader(&options)?;
assert_eq!(reader.schema().fields().len(), 1);
assert_eq!(reader.count(), 0);

// An empty write still publishes a readable file with the schema in its footer.
let mut media = Parquet::new(Buffer::new().with_media_type(MimeType::PARQUET.into()));
let options = media.record_options()?;
media.overwrite_arrow_reader(
    arrow::batch_reader(
        field.into_arrow_schema()?,
        std::iter::empty::<RecordBatch>(),
    ),
    &options,
)?;
assert_eq!(media.read_arrow_reader(&options)?.count(), 0);
assert_eq!(media.read_arrow_schema()?.fields().len(), 1);
assert_eq!(media.read_statistics()?.num_rows, 0);